# DevRev Search Dataset

Loading and exploring the `devrev/search` dataset from Hugging Face.

**Dataset Structure:**
- `annotated_queries` — Queries paired with annotated (golden) article chunks
- `knowledge_base` — Article chunks from DevRev's customer-facing support documentation
- `test_queries` — Held-out queries used for evaluation

In [ ]:
from datasets import load_dataset
import pandas as pd

## 1. Load Annotated Queries
Queries paired with annotated (golden) article chunks for training/validation.

In [ ]:
# Load annotated queries
annotated_queries = load_dataset("devrev/search", "annotated_queries")
print(annotated_queries)

In [ ]:
# Convert to DataFrame and display
annotated_df = annotated_queries["train"].to_pandas()
annotated_df.head()

In [ ]:
# Sample a single annotated query example
annotated_queries["train"][0]

## 2. Load Test Queries
Held-out queries used for evaluation.

In [ ]:
# Load test queries
test_queries = load_dataset("devrev/search", "test_queries")
print(test_queries)

In [ ]:
# Convert to DataFrame and display
test_df = test_queries["test"].to_pandas()
test_df.head()

In [ ]:
# Sample a single test query example
test_queries["test"][0]

## 3. Load Knowledge Base
Article chunks from DevRev's customer-facing support documentation.

In [ ]:
# Load knowledge base
knowledge_base = load_dataset("devrev/search", "knowledge_base")
print(knowledge_base)

In [ ]:
# Convert to DataFrame and display
knowledge_df = knowledge_base["corpus"].to_pandas()
knowledge_df.head()

In [ ]:
# Sample a single knowledge base chunk
knowledge_base["corpus"][0]

## 4. Dataset Summary

In [ ]:
print("=" * 60)
print("DevRev Search Dataset Summary")
print("=" * 60)
print(f"\nAnnotated Queries:")
print(annotated_queries)
print(f"\nTest Queries:")
print(test_queries)
print(f"\nKnowledge Base:")
print(knowledge_base)
print("\n" + "=" * 60)

---
## 5. Index Knowledge Base with FAISS

Using OpenAI text-embedding-3-small and FAISS for similarity search.

In [ ]:
from openai import OpenAI
import faiss
import numpy as np
from tqdm import tqdm
import time
import os
import json
import pickle
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential

In [ ]:
# Initialize OpenAI client
# Set your API key as an environment variable: export OPENAI_API_KEY="your-key-here"
%load_ext dotenv
%reload_ext dotenv
%dotenv
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Please set OPENAI_API_KEY environment variable")

client = AzureOpenAI(api_key=OPENAI_API_KEY, api_version="2024-02-01", azure_endpoint="https://foundry-adkumar.cognitiveservices.azure.com/")

MODEL_ID = "text-embedding-3-small"  # 1536 dimensions
print(f"Using model: {MODEL_ID}")
print(f"Provider: OpenAI")

In [ ]:
def get_embedding(text: str) -> np.ndarray:
    """Get embedding for a single text using OpenAI text-embedding-3."""
    response = client.embeddings.create(
        model=MODEL_ID,
        input=text
    )
    return np.array(response.data[0].embedding)

In [ ]:
def get_embeddings_batch(texts: list) -> np.ndarray:
    """Get embeddings for a batch of texts using OpenAI text-embedding-3."""
    response = client.embeddings.create(
        model=MODEL_ID,
        input=texts
    )
    return np.array([d.embedding for d in response.data])

In [ ]:
# Test the embedding API
test_embedding = get_embedding("This is a test sentence.")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

In [ ]:
# Prepare documents: concatenate title with text
corpus = knowledge_base["corpus"]

documents = []
doc_ids = []
doc_titles = []
doc_texts = []

for item in tqdm(corpus, desc="Preparing documents"):
    # Concatenate title and text
    doc_text = f"{item['title']}\n\n{item['text']}"
    documents.append(doc_text)
    doc_ids.append(item['id'])
    doc_titles.append(item['title'])
    doc_texts.append(item['text'])

print(f"\nTotal documents: {len(documents):,}")
print(f"\nSample document:")
print(documents[0][:500])

In [ ]:
def get_all_embeddings(texts, batch_size=100, max_retries=6):
    """
    Get embeddings for all texts using Azure OpenAI safely.
    Uses exponential backoff + jitter to prevent 429 storms.
    """
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch = texts[i:i + batch_size]

        # Optional: truncate overly long texts (better to tokenize in production)
        batch = [text[:8000] if len(text) > 8000 else text for text in batch]

        for attempt in range(max_retries):
            try:
                embeddings = get_embeddings_batch(batch)
                all_embeddings.append(embeddings)
                break

            # except RateLimitError as e:
            #     # Exponential backoff with jitter
            #     sleep_time = (2 ** attempt) + random.uniform(0, 1)
            #     print(f"429 hit. Retry {attempt + 1}/{max_retries} "
            #           f"for batch {i}. Sleeping {sleep_time:.2f}s")
            #     time.sleep(sleep_time)

            except Exception as e:
                # Other transient errors
                sleep_time = (2 ** attempt) + random.uniform(0, 1)
                print(f"Error: {e}. Retry {attempt + 1}/{max_retries}")
                time.sleep(sleep_time)

        else:
            # If all retries failed
            print(f"Failed embedding batch {i}. Filling with zeros.")
            all_embeddings.append(np.zeros((len(batch), 1536)))

        # Small pacing delay between batches (adjust as needed)
        time.sleep(0.5)

    return np.vstack(all_embeddings)

In [ ]:
# Generate embeddings for all documents
print("Generating embeddings for knowledge base...")
print(f"Total documents: {len(documents):,}")
print("Using batch processing for efficiency...")

# For testing, use a subset (uncomment for all documents)
# documents_to_embed = documents[:100]  # Test with first 100 docs
documents_to_embed = documents  # All documents

embeddings = get_all_embeddings(documents_to_embed, batch_size=1000)
print(f"\nEmbeddings shape: {embeddings.shape}")

# Save embeddings
np.save("embeddings.npy", embeddings)
print("Embeddings saved to embeddings.npy")

In [ ]:
# Load embeddings if already saved
# embeddings = np.load("embeddings.npy")
# print(f"Loaded embeddings shape: {embeddings.shape}")

In [ ]:
# Create FAISS index
embedding_dim = embeddings.shape[1]
print(f"Creating FAISS index with dimension: {embedding_dim}")

# Normalize embeddings for cosine similarity
embeddings_normalized = embeddings.copy().astype('float32')
faiss.normalize_L2(embeddings_normalized)

# Create the index using IndexFlatIP for inner product (cosine similarity with normalized vectors)
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings_normalized)

print(f"Index created with {index.ntotal:,} vectors")

In [ ]:
# Save the index and document mapping
INDEX_DIR = "faiss_index"
os.makedirs(INDEX_DIR, exist_ok=True)

# Save FAISS index
faiss.write_index(index, os.path.join(INDEX_DIR, "knowledge_base.index"))

# Save document mapping
with open(os.path.join(INDEX_DIR, "doc_mapping.pkl"), "wb") as f:
    pickle.dump({
        "doc_ids": doc_ids,
        "documents": documents,
        "doc_titles": doc_titles,
        "doc_texts": doc_texts
    }, f)

print(f"✓ Index saved to {INDEX_DIR}/knowledge_base.index")
print(f"✓ Document mapping saved to {INDEX_DIR}/doc_mapping.pkl")

## 6. Search the Knowledge Base

In [ ]:
def search(query: str, k: int = 5):
    """Search the knowledge base for relevant documents."""
    query_embedding = get_embedding(query).astype('float32')
    query_embedding = query_embedding.reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    scores, indices = index.search(query_embedding, k)
    
    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append({
            "rank": i + 1,
            "score": float(score),
            "id": doc_ids[idx],
            "title": doc_titles[idx],
            "text": doc_texts[idx]
        })
    
    return results

In [ ]:
# Test search with a sample query
query = "How do I set up AirSync?"
results = search(query, k=5)

print(f"Query: {query}")
print("=" * 60)

for r in results:
    print(f"\n[Rank {r['rank']}] Score: {r['score']:.4f}")
    print(f"Doc ID: {r['id']}")
    print(f"Title: {r['title']}")
    print(f"Text: {r['text'][:300]}...")
    print("-" * 40)

---
## 7. Run Evaluation on Test Queries

Run search against all test queries and save results in the same format as annotated_queries.

In [ ]:
# Run search on all test queries - DIRECT VERSION (no dependency on search function)
TOP_K = 10  # Number of retrievals per query

# Get corpus data for lookups
corpus_data = knowledge_base["corpus"]

test_results = []

for item in tqdm(test_queries["test"], desc="Processing test queries"):
    query_id = item["query_id"]
    query = item["query"]
    
    # Get query embedding directly
    query_embedding = get_embedding(query).astype('float32').reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    # Search FAISS index
    scores, indices = index.search(query_embedding, TOP_K)
    
    # Format retrievals using corpus data directly
    retrievals = []
    for idx in indices[0]:
        doc = corpus_data[int(idx)]
        retrievals.append({
            "id": doc["id"],
            "text": doc["text"],
            "title": doc["title"]
        })
    
    test_results.append({
        "query_id": query_id,
        "query": query,
        "retrievals": retrievals
    })

print(f"\nProcessed {len(test_results)} test queries")

In [ ]:
# Preview a sample result
import json
print("Sample result:")
print(json.dumps(test_results[0], indent=2, default=str)[:1500])

In [ ]:
# Save results to JSON file
OUTPUT_FILE = "test_queries_results.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(test_results, f, indent=2)

print(f"✓ Results saved to {OUTPUT_FILE}")
print(f"  - {len(test_results)} queries")
print(f"  - {TOP_K} retrievals per query")

In [ ]:
# Also save as a parquet file for easier loading
results_df = pd.DataFrame(test_results)
results_df.to_parquet("test_queries_results.parquet", index=False)
print("✓ Results also saved to test_queries_results.parquet")

In [ ]:
# Display results summary
print("=" * 60)
print("Test Queries Results Summary")
print("=" * 60)
print(f"Total queries: {len(test_results)}")
print(f"Retrievals per query: {TOP_K}")
print(f"\nOutput files:")
print(f"  - test_queries_results.json")
print(f"  - test_queries_results.parquet")
print("\nFormat matches annotated_queries structure:")
print("  - query_id: string")
print("  - query: string")
print("  - retrievals: list of {id, text, title}")

## 8. Load Saved Index (Optional)
Use this to load a previously saved index without re-embedding.

In [ ]:
# Load saved index and mapping
# INDEX_DIR = "faiss_index"
# index = faiss.read_index(os.path.join(INDEX_DIR, "knowledge_base.index"))
# with open(os.path.join(INDEX_DIR, "doc_mapping.pkl"), "rb") as f:
#     mapping = pickle.load(f)
#     doc_ids = mapping["doc_ids"]
#     documents = mapping["documents"]
#     doc_titles = mapping["doc_titles"]
#     doc_texts = mapping["doc_texts"]
# print(f"Loaded index with {index.ntotal:,} vectors")

---
# Hybrid Search: Dense (OpenAI FAISS) + Sparse (BM25) + Hybrid Combiner

Reusing existing OpenAI `text-embedding-3-small` (1536-dim) embeddings and FAISS index.
Adding BM25 sparse retrieval for hybrid search.
The hybrid combiner fuses both using **Reciprocal Rank Fusion (RRF)** and **Weighted Sum**.

In [ ]:
# Section 9: Install additional dependencies
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "rank_bm25"])
print("✓ Dependencies installed")

In [ ]:
# Imports for hybrid search
from rank_bm25 import BM25Okapi
import faiss
import numpy as np
import re
import os
import json
import pickle
from tqdm import tqdm
from collections import defaultdict
print("✓ All imports successful")

---
## 9. Dense Retriever — Reuse Existing OpenAI Embeddings + FAISS

Loading the pre-built FAISS index and document mapping created with
OpenAI `text-embedding-3-small` (1536 dimensions).
No new embeddings needed — reusing what was generated in Section 5.

In [ ]:
# Load the existing FAISS index
INDEX_DIR = "faiss_index"
dense_index = faiss.read_index(os.path.join(INDEX_DIR, "knowledge_base.index"))
print(f"✓ FAISS index loaded with {dense_index.ntotal:,} vectors")
print(f"  Embedding dimension: {dense_index.d}")

In [ ]:
# Load document mapping
with open(os.path.join(INDEX_DIR, "doc_mapping.pkl"), "rb") as f:
    mapping = pickle.load(f)
    doc_ids_hybrid = mapping["doc_ids"]
    documents_hybrid = mapping["documents"]
    doc_titles_hybrid = mapping["doc_titles"]
    doc_texts_hybrid = mapping["doc_texts"]

print(f"✓ Document mapping loaded: {len(doc_ids_hybrid):,} documents")

In [ ]:
def dense_search(query: str, k: int = 100):
    '''Dense retrieval using existing OpenAI embeddings + FAISS index.
    Uses the get_embedding() function defined in Section 5 (OpenAI API).
    Returns list of (doc_index, score) sorted by descending score.
    '''
    # get_embedding() was defined in Section 5 — uses OpenAI text-embedding-3-small
    query_embedding = get_embedding(query).astype("float32").reshape(1, -1)
    faiss.normalize_L2(query_embedding)

    scores, indices = dense_index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:  # FAISS returns -1 for empty slots
            results.append((int(idx), float(score)))

    return results

# Quick test
test_results_dense = dense_search("How do I set up AirSync?", k=5)
print("Dense search test (top 5):")
for idx, score in test_results_dense:
    print(f"  [{score:.4f}] {doc_ids_hybrid[idx]} — {doc_titles_hybrid[idx][:60]}")

---
## 10. Sparse Retriever — BM25

Using BM25Okapi with simple whitespace + alphanumeric tokenization.
Great for exact keyword matching that dense models sometimes miss.

In [ ]:
def tokenize(text: str) -> list:
    '''Simple, consistent tokenization: lowercase + split on non-alphanumeric.'''
    return re.findall(r"[a-z0-9]+", text.lower())

# Tokenize all documents
print("Tokenizing documents for BM25...")
tokenized_docs = [tokenize(doc) for doc in tqdm(documents_hybrid, desc="Tokenizing")]
print(f"✓ Tokenized {len(tokenized_docs):,} documents")
print(f"  Average tokens per doc: {np.mean([len(d) for d in tokenized_docs]):.0f}")

In [ ]:
# Build BM25 index
print("Building BM25 index...")
bm25 = BM25Okapi(tokenized_docs)
print("✓ BM25 index built")

In [ ]:
def bm25_search(query: str, k: int = 100):
    '''Sparse retrieval using BM25.
    Returns list of (doc_index, score) sorted by descending score.
    '''
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)

    # Get top-k indices
    top_k_indices = np.argsort(scores)[-k:][::-1]

    results = []
    for idx in top_k_indices:
        if scores[idx] > 0:  # Only include docs with non-zero BM25 score
            results.append((int(idx), float(scores[idx])))

    return results

# Quick test
test_results_bm25 = bm25_search("How do I set up AirSync?", k=5)
print("BM25 search test (top 5):")
for idx, score in test_results_bm25:
    print(f"  [{score:.4f}] {doc_ids_hybrid[idx]} — {doc_titles_hybrid[idx][:60]}")

---
## 11. Hybrid Search — Combining Dense + Sparse

Two fusion strategies:
1. **Weighted Sum** — normalize scores to [0,1] then combine: `alpha * dense + (1-alpha) * bm25`
2. **Reciprocal Rank Fusion (RRF)** — rank-based, no score calibration needed: `sum(1 / (k + rank))`

In [ ]:
def min_max_normalize(scores_list):
    '''Normalize a list of (idx, score) to [0, 1] range using min-max.'''
    if not scores_list:
        return []
    scores = np.array([s for _, s in scores_list])
    min_s, max_s = scores.min(), scores.max()
    if max_s - min_s < 1e-9:
        # All scores identical — assign uniform score
        return [(idx, 1.0) for idx, _ in scores_list]
    normalized = (scores - min_s) / (max_s - min_s)
    return [(idx, float(ns)) for (idx, _), ns in zip(scores_list, normalized)]

In [ ]:
def hybrid_search_weighted(query: str, k: int = 10, alpha: float = 0.7,
                           retrieval_pool: int = 100):
    '''Hybrid search using weighted sum of normalized scores.
    alpha=0.7 means 70% dense + 30% BM25 (tune to taste).
    '''
    # Get top results from both retrievers
    dense_results = dense_search(query, k=retrieval_pool)
    bm25_results = bm25_search(query, k=retrieval_pool)

    # Normalize scores to [0, 1]
    dense_norm = min_max_normalize(dense_results)
    bm25_norm = min_max_normalize(bm25_results)

    # Merge scores
    combined = defaultdict(float)
    for idx, score in dense_norm:
        combined[idx] += alpha * score
    for idx, score in bm25_norm:
        combined[idx] += (1 - alpha) * score

    # Sort by combined score descending
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]

    return [(idx, score) for idx, score in ranked]

In [ ]:
def hybrid_search_rrf(query: str, k: int = 10, rrf_k: int = 60,
                      retrieval_pool: int = 100):
    '''Hybrid search using Reciprocal Rank Fusion (RRF).
    RRF score = sum over retrievers of 1/(rrf_k + rank).
    rrf_k=60 is the standard default from the original RRF paper.
    '''
    dense_results = dense_search(query, k=retrieval_pool)
    bm25_results = bm25_search(query, k=retrieval_pool)

    combined = defaultdict(float)

    # Dense RRF scores
    for rank, (idx, _) in enumerate(dense_results, start=1):
        combined[idx] += 1.0 / (rrf_k + rank)

    # BM25 RRF scores
    for rank, (idx, _) in enumerate(bm25_results, start=1):
        combined[idx] += 1.0 / (rrf_k + rank)

    # Sort by RRF score descending
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]

    return [(idx, score) for idx, score in ranked]

In [ ]:
# ── Side-by-side comparison on a sample query ──
sample_query = "How do I set up AirSync?"
print(f"Query: {sample_query}")
print("=" * 80)

for method_name, search_fn in [
    ("Dense only (OpenAI)", lambda q: dense_search(q, k=10)),
    ("BM25 only",  lambda q: bm25_search(q, k=10)),
    ("Hybrid (Weighted α=0.7)", lambda q: hybrid_search_weighted(q, k=10, alpha=0.7)),
    ("Hybrid (RRF k=60)",       lambda q: hybrid_search_rrf(q, k=10, rrf_k=60)),
]:
    results = search_fn(sample_query)
    print(f"\n── {method_name} ──")
    for rank, (idx, score) in enumerate(results[:5], 1):
        print(f"  {rank}. [{score:.4f}] {doc_ids_hybrid[idx]} — {doc_titles_hybrid[idx][:55]}")
    if len(results) > 5:
        print(f"  ... and {len(results) - 5} more")

---
## 12. Evaluate Hybrid Search on Test Queries

Run hybrid search (RRF) on all 92 test queries and save results in the same format as the original evaluation.

In [ ]:
# Run hybrid (RRF) search on all test queries
TOP_K_HYBRID = 10

hybrid_test_results = []

for item in tqdm(test_queries["test"], desc="Hybrid search on test queries"):
    query_id = item["query_id"]
    query = item["query"]

    # Use RRF hybrid search
    hybrid_results = hybrid_search_rrf(query, k=TOP_K_HYBRID, rrf_k=60, retrieval_pool=100)

    # Format retrievals
    retrievals = []
    for idx, score in hybrid_results:
        retrievals.append({
            "id": doc_ids_hybrid[idx],
            "text": doc_texts_hybrid[idx],
            "title": doc_titles_hybrid[idx]
        })

    hybrid_test_results.append({
        "query_id": query_id,
        "query": query,
        "retrievals": retrievals
    })

print(f"\n✓ Processed {len(hybrid_test_results)} test queries")

In [ ]:
# Save hybrid results
HYBRID_OUTPUT_FILE = "hybrid_test_queries_results.json"

with open(HYBRID_OUTPUT_FILE, "w") as f:
    json.dump(hybrid_test_results, f, indent=2)

print(f"✓ Hybrid results saved to {HYBRID_OUTPUT_FILE}")
print(f"  - {len(hybrid_test_results)} queries")
print(f"  - {TOP_K_HYBRID} retrievals per query")
print(f"  - Method: RRF (k=60), pool=100 from each retriever")

In [ ]:
# ── Summary comparison ──
print("=" * 60)
print("Hybrid Search Pipeline Summary")
print("=" * 60)
print(f"\nDense Retriever:")
print(f"  Model      : OpenAI text-embedding-3-small (reused)")
print(f"  Dimensions : {dense_index.d}")
print(f"  Index size : {dense_index.ntotal:,} vectors")
print(f"  Source     : faiss_index/knowledge_base.index")
print(f"\nSparse Retriever:")
print(f"  Method     : BM25Okapi")
print(f"  Corpus     : {len(tokenized_docs):,} documents")
print(f"  Avg tokens : {np.mean([len(d) for d in tokenized_docs]):.0f} per doc")
print(f"\nHybrid Combiner:")
print(f"  Method     : Reciprocal Rank Fusion (RRF, k=60)")
print(f"  Pool size  : 100 from each retriever")
print(f"  Final top-K: {TOP_K_HYBRID}")
print(f"\nOutput: {HYBRID_OUTPUT_FILE}")
print("=" * 60)